In [7]:
import pandas as pd

In [8]:
df = pd.read_csv('../data/raw/61111-0002_en_flat.csv',
                 sep=';')
cpi = df[df['value_unit'] == '2020=100'].copy()
roi = pd.read_csv('../data/processed/01_roi_comparison.csv')

print(cpi.head())
print(roi.head())

    statistics_code                  statistics_label time_code time_label  \
1             61111  Consumer price index for Germany      JAHR       Year   
4             61111  Consumer price index for Germany      JAHR       Year   
7             61111  Consumer price index for Germany      JAHR       Year   
10            61111  Consumer price index for Germany      JAHR       Year   
13            61111  Consumer price index for Germany      JAHR       Year   

    time 1_variable_code 1_variable_label 1_variable_attribute_code  \
1   2019           MONAT           Months                   MONAT05   
4   2025           MONAT           Months                   MONAT07   
7   2020           MONAT           Months                   MONAT05   
10  2017           MONAT           Months                   MONAT10   
13  2019           MONAT           Months                   MONAT06   

   1_variable_attribute_label 2_variable_code 2_variable_label  \
1                         May         

In [9]:
cpi["month"] = (
    cpi["1_variable_attribute_code"]
    .str.replace("MONAT", "", regex=False)
    .astype(int)
)

cpi["Date"] = pd.to_datetime(
    dict(
        year=cpi["time"],
        month=cpi["month"],
        day=1
    )
)

cpi = cpi[["Date", "value"]]
cpi['value'] = pd.to_numeric(cpi['value'])
cpi = cpi.sort_values('Date')

cpi['Date'] = pd.to_datetime(cpi['Date'])
cpi = cpi.set_index('Date')

cpi_q = cpi.resample('QE').mean().reset_index()
cpi_q['period'] = cpi_q['Date'].dt.to_period('Q')
cpi_q = cpi_q.drop(columns='Date')

cpi_q = cpi_q.rename(columns={'value': 'cpi_index'})
roi['period'] = pd.PeriodIndex(roi['period'], freq='Q')
cpi_q = cpi_q[['period', 'cpi_index']]

print(cpi_q.head())

   period  cpi_index
0  2016Q1  94.000000
1  2016Q2  94.966667
2  2016Q3  95.466667
3  2016Q4  95.400000
4  2017Q1  95.500000


In [10]:
merged = pd.merge(cpi_q, roi, on='period', how='inner')
merged['cpi_index_norm'] = (merged['cpi_index'] / merged['cpi_index'].iloc[0]) * 100
merged['period'] = merged['period'].dt.to_timestamp('Q')

print(merged.head())

      period  cpi_index    EUNL.DE        ^GSPC  house_price_index  \
0 2016-03-31  94.000000  35.990002  2059.739990              103.9   
1 2016-06-30  94.966667  37.139999  2098.860107              106.9   
2 2016-09-30  95.466667  38.709999  2168.270020              108.8   
3 2016-12-31  95.400000  42.180000  2238.830078              110.4   
4 2017-03-31  95.500000  44.070000  2362.719971              110.9   

   new_construction_index  existing_property_index  EUNL.DE_norm  ^GSPC_norm  \
0                   104.2                    103.8    100.000000  100.000000   
1                   106.2                    107.0    103.195326  101.899275   
2                   107.5                    109.0    107.557647  105.269113   
3                   109.5                    110.5    117.199217  108.694791   
4                   109.3                    111.2    122.450674  114.709623   

   house_price_index_norm  new_construction_index_norm  \
0              100.000000               

In [11]:
merged.to_csv('../data/processed/02_roi_with_macro.csv', index=False)